# AC-Zero — Dataset Generation (Kaggle)

Grows a persistent, guaranteed-solvable Andrews-Curtis dataset outward from the
trivial group with `aczero dataset grow`, until the Kaggle time budget is nearly
spent, then writes the dataset **and** a statistics summary to `/kaggle/working`
(the notebook's saved output).

**Before you run**
- Settings -> **Internet: On** (needed to `pip install` from GitHub).
- Accelerator: **None / CPU** is enough — generation is CPU graph search, no GPU.
- Kaggle sessions are capped at ~12 h. `TIME_BUDGET_HOURS` below stops the grow
  with a safety margin so the dataset is always flushed to disk before the kill.
- Grow is **resumable via the Hugging Face bucket**: the dataset is pulled at
  start and pushed back at the end, so each session continues the last.


## 1. Configuration

In [ ]:
import os

# --- Repository -------------------------------------------------------------
REPO_URL = "https://github.com/HkHk2Prod/AlphaAC.git"
REPO_BRANCH = "main"  # branch / tag / commit to install

# --- Dataset shape ----------------------------------------------------------
RANK = 2  # group rank (catalog has 3*rank^2 primitive moves)
TOTAL_LENGTH_CAP = 48  # discard presentations longer than this
SELECT = "smallest"  # frontier order: "smallest" | "weighted-random"
SEED = 0
WORKERS = 0  # 0 = all CPU cores; 1 = single in-process worker

# --- Time budget (Kaggle hard-kills at ~12 h) -------------------------------
TIME_BUDGET_HOURS = 11.5  # stop growing after this much wall-clock
SAFETY_MARGIN_MIN = 10  # head-room reserved for the final flush + summary

# --- Growth loop ------------------------------------------------------------
TARGET_PER_CHUNK = 2000  # new groups added per grow() call
CHECKPOINT_EVERY = 1000  # grow() flushes to disk every N new groups

# --- Output / resume --------------------------------------------------------
OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATASET_NAME = f"train_rank{RANK}.groups.json"
RESUME_FROM = ""  # e.g. "/kaggle/input/ac-zero-dataset/train_rank2.json"

# --- Annotation pass: per-move-set distances --------------------------------
# `dataset annotate` writes a companion <name>.<moveset>.annotations.json with
# each group's distance to the origin (trivial group) and its length-descent
# distance. Training's `descent` reward (02_train REWARD_MODE = "descent") reads
# the descent distance as N from the strict-ac annotation (the env applies strict
# AC moves). List the move sets to annotate; an empty list skips the pass.
ANNOTATE_MOVESETS = ["universal", "strict-ac"]

# --- Hugging Face bucket (dataset storage; too big for GitHub) ---------------
# The dataset lives in a HF bucket instead of git. On start the notebook pulls
# the current dataset (resume); on finish it pushes the grown one back, so each
# session continues where the last left off. Add an `HF_TOKEN` Kaggle secret
# (Add-ons -> Secrets) with write access to the bucket namespace.
HF_BUCKET = "HkHk2Prod/alphaac-data"  # namespace/bucket holding the datasets
HF_REMOTE_NAME = DATASET_NAME  # object name inside the bucket
HF_DOWNLOAD_ON_START = True  # resume by pulling the current dataset from the bucket
HF_UPLOAD_ON_FINISH = True  # push the grown dataset back to the bucket when done


## 2. Install AC-Zero from GitHub

In [ ]:
import subprocess
import sys


# --no-deps keeps Kaggle's pre-built torch/numpy; we add only the light deps the
# generation path actually needs (the grow search itself is pure-Python).
def pip_install(*args):
    # Quiet install: capture pip's output and surface it only on real failure.
    # Kaggle's base image ships a broken google-adk dependency, so pip prints a
    # "dependency resolver" ERROR on every run even when our install succeeds;
    # capturing keeps that unrelated noise (and download progress bars) out of
    # the log while still raising if the install itself fails.
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", *args],
        capture_output=True,
        text=True,
    )
    if proc.returncode:
        print(proc.stdout, proc.stderr, sep="\n")
        proc.check_returncode()

spec = f"git+{REPO_URL}@{REPO_BRANCH}"
pip_install("--no-deps", spec)
pip_install(
    "numpy>=1.26",
    "pydantic>=2",
    "pyyaml>=6",
    "typer>=0.12",
    "rich>=13",
    "gymnasium>=0.29",
    "matplotlib>=3.8",
    "huggingface_hub>=1.21",
)

print("Installed ac_zero from", spec)


## 3. Time budget

In [ ]:
import time

START_TIME = time.monotonic()
DEADLINE = START_TIME + TIME_BUDGET_HOURS * 3600
MARGIN_S = SAFETY_MARGIN_MIN * 60


def elapsed_s():
    return time.monotonic() - START_TIME


def time_left_s():
    return DEADLINE - time.monotonic()


def hms(sec):
    sec = max(0, int(sec))
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h:d}h{m:02d}m{s:02d}s"


class DeadlineReached(Exception):
    """Raised from a progress callback to stop cleanly before the Kaggle kill."""


print(
    f"Budget {TIME_BUDGET_HOURS} h with a {SAFETY_MARGIN_MIN} min safety margin "
    f"(stop growing when {hms(MARGIN_S)} remain)."
)

## 4. Resume from the Hugging Face bucket

Pull the current dataset from the bucket so this session continues growing it.
The first ever run finds nothing and starts fresh. Needs an `HF_TOKEN` secret.

In [ ]:
import os
from pathlib import Path

from ac_zero.datasets.hub import download_dataset

# Pick up the HF token from a Kaggle secret if it is not already in the env.
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass  # no HF_TOKEN secret attached to this kernel

if HF_UPLOAD_ON_FINISH and not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        "HF_UPLOAD_ON_FINISH is True but no HF_TOKEN is available (env var or "
        "Kaggle secret) -- growing would run for the whole time budget and then "
        "fail to publish the result. Add an HF_TOKEN Kaggle secret (Add-ons -> "
        "Secrets) with write access, or set HF_UPLOAD_ON_FINISH = False."
    )

out_path = Path(OUTPUT_DIR) / DATASET_NAME
out_path.parent.mkdir(parents=True, exist_ok=True)

if HF_DOWNLOAD_ON_START:
    try:
        pulled = download_dataset(
            out_path, remote_name=HF_REMOTE_NAME, bucket=HF_BUCKET, missing_ok=True
        )
        if pulled is None:
            print("No dataset in the bucket yet - starting a fresh graph.")
        else:
            print(f"Resumed {pulled} ({pulled.stat().st_size / 1e6:.1f} MB) "
                  f"from hf://buckets/{HF_BUCKET}/{HF_REMOTE_NAME}")
    except Exception as exc:
        print("Bucket download skipped:", exc)


## 5. Grow the dataset

In [ ]:
import shutil
from pathlib import Path

from ac_zero.datasets.grow import GrowConfig, grow_dataset

out_path = Path(OUTPUT_DIR) / DATASET_NAME
out_path.parent.mkdir(parents=True, exist_ok=True)

# Optional resume: seed the working file from a previous session's output; grow
# then continues from the accumulated frontier and only ever adds groups.
if RESUME_FROM and Path(RESUME_FROM).exists() and Path(RESUME_FROM).resolve() != out_path.resolve():
    shutil.copy(RESUME_FROM, out_path)
    print("Resuming from", RESUME_FROM)


def progress(message, metrics):
    # grow() checkpoints between rounds, so raising here loses at most the groups
    # added since the last checkpoint — the dataset on disk stays consistent.
    if time_left_s() <= MARGIN_S:
        raise DeadlineReached(message)
    if message in ("checkpoint", "grow complete") or "added" in metrics:
        print(f"[{hms(elapsed_s())}] {message}: {metrics}")


total_added = 0
chunks = 0
try:
    while time_left_s() > MARGIN_S:
        cfg = GrowConfig(
            rank=RANK,
            target=TARGET_PER_CHUNK,
            select=SELECT,
            seed=SEED,
            total_length_cap=TOTAL_LENGTH_CAP,
            workers=WORKERS,
            checkpoint_every=CHECKPOINT_EVERY,
            log_every=max(1, TARGET_PER_CHUNK // 4),
        )
        report = grow_dataset(out_path, cfg, progress=progress)
        total_added += report.added
        chunks += 1
        print(
            f"[{hms(elapsed_s())}] chunk {chunks}: +{report.added} groups "
            f"(total {report.total}, frontier {report.frontier}, max length {report.max_length})"
        )
        if report.added == 0:  # frontier exhausted within the length cap
            print("Frontier exhausted within the length cap — nothing left to add.")
            break
except DeadlineReached as stop:
    print(f"[{hms(elapsed_s())}] time budget reached ({stop}); dataset flushed to disk.")

print(f"\nAdded ~{total_added} groups across {chunks} chunk(s) in {hms(elapsed_s())}.")
print("Dataset:", out_path, f"({out_path.stat().st_size / 1e6:.1f} MB)")

## 6. Optional — per-move-set distance annotation

Annotate the grown group dataset with `aczero dataset annotate`, writing a
companion `<name>.<moveset>.annotations.json` per move set. Each entry carries the
group's distance to the origin (the trivial group) and its length-descent
distance. The `descent` reward in `02_train` (`REWARD_MODE = "descent"`) reads the
descent distance as its minimal-moves N from the **strict-ac** annotation. Set the
move sets to run in `ANNOTATE_MOVESETS` (config cell); the pass is time-boxed like
the grow.

In [ ]:
from ac_zero.datasets.annotate import AnnotateConfig, annotate

annotation_paths = []
if not ANNOTATE_MOVESETS:
    print("Skipping annotation (ANNOTATE_MOVESETS is empty).")
for moveset in ANNOTATE_MOVESETS:
    acfg = AnnotateConfig(moveset=moveset, max_depth=32, workers=WORKERS,
                          checkpoint_every=CHECKPOINT_EVERY)

    def aprogress(message, metrics, moveset=moveset):
        if time_left_s() <= MARGIN_S:
            raise DeadlineReached(message)
        print(f"[{hms(elapsed_s())}] {moveset} {message}: {metrics}")

    try:
        from ac_zero.datasets.annotate import annotation_path
        arep = annotate(out_path, acfg, progress=aprogress)
        apath = annotation_path(out_path, moveset)
        annotation_paths.append(apath)
        print(f"annotate[{moveset}]:", arep)
    except DeadlineReached:
        from ac_zero.datasets.annotate import annotation_path
        annotation_paths.append(annotation_path(out_path, moveset))
        print(f"{moveset} annotation stopped at the time budget (partial, checkpointed).")
        break


## 7. Dataset summary & output artifacts

In [ ]:
import json
from collections import Counter

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

data = json.loads(out_path.read_text())
groups = data.get("groups", [])


def dist(values):
    counts = Counter(values)
    pop = sum(counts.values())
    mean = (sum(v * n for v, n in counts.items()) / pop) if pop else None
    return {
        "counts": dict(sorted(counts.items())),
        "population": pop,
        "min": min(counts) if counts else None,
        "max": max(counts) if counts else None,
        "mean": mean,
    }


lengths = [int(e.get("total_length", 0)) for e in groups]
degrees = [len(e["transitions"]) for e in groups if e.get("transitions") is not None]
frontier = sum(1 for e in groups if e.get("transitions") is None)
by_source = dict(Counter(str(e.get("source", "unknown")) for e in groups).most_common())
ac_trivial = sum(1 for e in groups if e.get("ac_trivial") is True)

stats = {
    "rank": data.get("rank"),
    "schema_version": data.get("schema_version"),
    "move_catalog": data.get("move_catalog"),
    "generator": (data.get("provenance") or {}).get("generator"),
    "total_groups": len(groups),
    "frontier_open": frontier,
    "expanded": len(groups) - frontier,
    "ac_trivial": ac_trivial,
    "by_source": by_source,
    "generated_seconds": round(elapsed_s(), 1),
    "total_length": dist(lengths),
    "transition_degree": dist(degrees),
}
(Path(OUTPUT_DIR) / "dataset_stats.json").write_text(json.dumps(stats, indent=2))


def table(title, unit, d):
    lines = [f"### {title}", ""]
    if not d["population"]:
        return lines + ["_No data._", ""]
    lines.append(f"- min {d['min']} · max {d['max']} · mean {d['mean']:.2f}")
    lines += ["", f"| {unit} | groups |", "| --- | --- |"]
    lines += [f"| {k} | {v} |" for k, v in d["counts"].items()]
    return lines + [""]


report_md = [
    f"# Dataset summary: {DATASET_NAME}",
    "",
    f"- Generator: `{stats['generator']}`",
    f"- Schema: `{stats['schema_version']}` · Moves: `{stats['move_catalog']}`",
    f"- Rank: {stats['rank']}",
    f"- Total groups: {stats['total_groups']}",
    f"- Frontier (unexpanded): {stats['frontier_open']} · Expanded: {stats['expanded']}",
    f"- AC-trivial: {stats['ac_trivial']}",
    f"- Wall-clock: {hms(elapsed_s())}",
    "",
    "### By source",
    "",
    "| source | groups |",
    "| --- | --- |",
    *[f"| {s} | {n} |" for s, n in by_source.items()],
    "",
]
report_md += table("By size (total relator length)", "length", stats["total_length"])
report_md += table("By transition degree (universal neighbours in cap)", "degree", stats["transition_degree"])
(Path(OUTPUT_DIR) / "dataset_summary.md").write_text("\n".join(report_md) + "\n")

for key, label, fname in [
    ("total_length", "total relator length", "hist_length.png"),
    ("transition_degree", "transition degree", "hist_degree.png"),
]:
    d = stats[key]
    if d["population"]:
        fig, ax = plt.subplots(figsize=(6, 3.5))
        ax.bar(list(d["counts"].keys()), list(d["counts"].values()))
        ax.set_xlabel(label)
        ax.set_ylabel("groups")
        ax.set_title(f"Groups by {label}")
        fig.tight_layout()
        fig.savefig(Path(OUTPUT_DIR) / fname, dpi=120)
        plt.close(fig)

display(Markdown("\n".join(report_md)))
print("\nOutput artifacts in", OUTPUT_DIR, ":")
for p in sorted(Path(OUTPUT_DIR).glob("*")):
    if p.is_file():
        print(f"  {p.name:28s} {p.stat().st_size / 1e3:9.1f} KB")


## 8. Publish the dataset to the Hugging Face bucket

Push the grown dataset back so the next session (or `aczero dataset download`)
picks it up. Skipped if `HF_UPLOAD_ON_FINISH = False`.

In [ ]:
if HF_UPLOAD_ON_FINISH:
    from ac_zero.datasets.hub import upload_dataset

    # Push the group dataset and every annotation file (each syncs by basename).
    to_upload = [(out_path, HF_REMOTE_NAME)] + [(p, p.name) for p in annotation_paths if p.exists()]
    for path, remote in to_upload:
        try:
            uri = upload_dataset(path, remote_name=remote, bucket=HF_BUCKET)
            print("Uploaded", path.name, "to", uri)
        except Exception as exc:
            print("Bucket upload skipped for", path.name, ":", exc)
else:
    print("HF_UPLOAD_ON_FINISH is False - dataset left in", OUTPUT_DIR)
